# Multi-agent, guardrails & human approval

**Week 11 · Session 2 · Notebook 2 of 3**

Yesterday's concierge could search everything and was polite about it. It also:

- offered to **book** even though its rules said it couldn't,
- would happily accept a **passport number** pasted into the chat,
- would write your **college essay** if you asked nicely,
- carried **every tool** in one agent, a list that only grows.

Today we fix all four, and each fix is a different SDK primitive:

| Problem | Primitive |
|---|---|
| one agent doing everything | **handoffs** to specialist desks |
| instructions aren't enforcement | **guardrails**: input, output and tool |
| a booking is irreversible | **human approval**: pause, approve, resume |

This notebook is self-contained: it rebuilds yesterday's travel data and tools first.

In [ ]:
%pip install -q "openai-agents==0.22.3" python-dotenv

In [ ]:
import os, re, json, getpass
from pathlib import Path
from datetime import date, datetime

try:
    from dotenv import load_dotenv, find_dotenv
    load_dotenv(find_dotenv(usecwd=True) or Path.cwd().parent / ".env")
except ImportError:
    pass
if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass.getpass("OpenAI API key: ")

from pydantic import BaseModel
from agents import (Agent, Runner, function_tool, handoff, ModelSettings, ItemHelpers,
                    RunContextWrapper, RunState, SQLiteSession, trace,
                    input_guardrail, output_guardrail, GuardrailFunctionOutput,
                    InputGuardrailTripwireTriggered, OutputGuardrailTripwireTriggered,
                    tool_input_guardrail, ToolGuardrailFunctionOutput)
from agents.extensions.handoff_prompt import RECOMMENDED_PROMPT_PREFIX

# httpx2 2.12+ decompresses brotli with output_buffer_limit=... as a keyword.
# Anaconda's brotli 1.x exposes process() as a C function that rejects kwargs
# ("TypeError: process() takes no keyword arguments"). Wrap it so OpenAI
# Responses (which often arrive Content-Encoding: br) can be decoded.
try:
    import brotli as _brotli
    from httpx2._decoders import BrotliDecoder as _BrotliDecoder
    _probe = _brotli.Decompressor()
    try:
        _probe.process(b"", output_buffer_limit=1)
    except TypeError:
        _brotli_init = _BrotliDecoder.__init__
        def _brotli_init_compat(self, *args, **kwargs):
            _brotli_init(self, *args, **kwargs)
            _impl = self._decompress
            self._decompress = lambda data, output_buffer_limit=None, **_k: _impl(data)
        _BrotliDecoder.__init__ = _brotli_init_compat
except Exception:
    pass

MODEL = "gpt-4.1-mini"
TODAY = date.today().isoformat()
print("ready ·", TODAY)

In [ ]:
def show_run(result, width=105):
    """ACTION / OBSERVATION / HANDOFF / PENDING / ANSWER, plus cost."""
    for item in result.new_items:
        kind = type(item).__name__
        raw = getattr(item, "raw_item", None)
        if kind == "ToolCallItem":
            name = getattr(raw, "name", None) or getattr(raw, "type", "tool")
            print(f"  ACTION       {name}({(getattr(raw, 'arguments', '') or '')[:width]})")
        elif kind == "ToolCallOutputItem":
            print(f"  OBSERVATION  {str(item.output)[:width]}")
        elif kind == "HandoffOutputItem":
            print(f"  HANDOFF      {item.source_agent.name}  ->  {item.target_agent.name}")
        elif kind == "ToolApprovalItem":
            print(f"  APPROVAL     {raw.name}({raw.arguments})   <- gated: needs a human")
        elif kind == "MessageOutputItem":
            print(f"  ANSWER       {ItemHelpers.text_message_output(item)[:width]}")
    u = result.context_wrapper.usage
    print(f"\n  {u.requests} model call(s) · {u.input_tokens} in + {u.output_tokens} out"
          f" · last agent: {result.last_agent.name}")

### The travel inventory and search tools from yesterday

Same demo data, same tools. **The visa rules are illustrative, not current policy.**

In [ ]:
FLIGHTS = [
    {"id": "6E-512",  "from": "BLR", "to": "GOI", "date": "2026-10-02", "depart": "07:10", "airline": "IndiGo",             "price_inr": 4180},
    {"id": "AI-887",  "from": "BLR", "to": "GOI", "date": "2026-10-02", "depart": "13:45", "airline": "Air India",          "price_inr": 5620},
    {"id": "QP-1375", "from": "BLR", "to": "GOI", "date": "2026-10-02", "depart": "18:30", "airline": "Akasa Air",          "price_inr": 3890},
    {"id": "EK-569",  "from": "BLR", "to": "DXB", "date": "2026-10-02", "depart": "04:30", "airline": "Emirates",           "price_inr": 21400},
    {"id": "6E-1485", "from": "BLR", "to": "DXB", "date": "2026-10-02", "depart": "22:05", "airline": "IndiGo",             "price_inr": 16800},
    {"id": "SQ-511",  "from": "BLR", "to": "SIN", "date": "2026-10-02", "depart": "23:55", "airline": "Singapore Airlines", "price_inr": 24900},
    {"id": "6E-1005", "from": "BLR", "to": "SIN", "date": "2026-10-02", "depart": "01:20", "airline": "IndiGo",             "price_inr": 18200},
]
HOTELS = {
    "goa":       [{"id": "H-GOA-1", "name": "Sea Breeze Candolim",      "area": "Candolim beach", "price_inr": 5200},
                  {"id": "H-GOA-2", "name": "Fontainhas Heritage Stay", "area": "Panjim",         "price_inr": 3800},
                  {"id": "H-GOA-3", "name": "Anjuna Cliff Resort",      "area": "Anjuna",         "price_inr": 9800}],
    "dubai":     [{"id": "H-DXB-1", "name": "Marina View Hotel",        "area": "Dubai Marina",   "price_inr": 11200},
                  {"id": "H-DXB-2", "name": "Deira Budget Inn",         "area": "Deira",          "price_inr": 5600}],
    "singapore": [{"id": "H-SIN-1", "name": "Bugis Boutique",           "area": "Bugis",          "price_inr": 12800},
                  {"id": "H-SIN-2", "name": "Little India Lodge",       "area": "Little India",   "price_inr": 6900}],
}
VISA_RULES = {   # DEMO DATA -- always verify with the embassy
    ("india", "goa"):       "Domestic travel. No visa. Carry a government photo ID.",
    ("india", "dubai"):     "Visa required for most Indian passport holders (e-visa).",
    ("india", "singapore"): "Visa required. Apply online through an authorised agent, 3-5 working days.",
}
BOOKINGS = []    # our pretend booking system


@function_tool
def search_flights(origin: str, destination: str, date: str) -> str:
    """Search flights between two airports on a date, cheapest first.

    Args:
        origin: IATA code, e.g. "BLR".
        destination: IATA code, e.g. "GOI".
        date: YYYY-MM-DD.
    """
    hits = sorted((f for f in FLIGHTS if f["from"] == origin.upper()
                   and f["to"] == destination.upper() and f["date"] == date), key=lambda f: f["price_inr"])
    return json.dumps(hits) if hits else f"No flights {origin}->{destination} on {date}."


@function_tool
def search_hotels(city: str, max_price_inr: int = 100000) -> str:
    """Find hotels in a city under a nightly budget, cheapest first.

    Args:
        city: e.g. "Goa".
        max_price_inr: maximum price per night.
    """
    opts = [h for h in HOTELS.get(city.strip().lower(), []) if h["price_inr"] <= max_price_inr]
    return json.dumps(sorted(opts, key=lambda h: h["price_inr"])) if opts else f"Nothing in {city} under INR {max_price_inr}."


@function_tool
def check_visa(passport_country: str, destination: str) -> str:
    """Visa requirements for a passport holder travelling to a destination city."""
    return VISA_RULES.get((passport_country.strip().lower(), destination.strip().lower()),
                          "No rule on file. Advise checking the embassy.")
print("tools ready")

---
# Part 1 — Why split one agent into several?

Yesterday's concierge had every tool. That works at five tools. At fifty it breaks, for
reasons you already know from Week 10:

- **Tool choice degrades.** More tools means more similar descriptions to confuse.
- **Context bloats.** Every tool schema is re-sent on every turn.
- **Instructions conflict.** "Be brief" for flights, "be thorough" for visas, in one prompt.
- **Ownership blurs.** When it's wrong, whose prompt do you fix?

The fix is the multi-agent pattern from Week 10's ladder: small specialists, each with a
narrow prompt and only the tools it needs.

---
# Part 2 — Handoffs

A **handoff** transfers the conversation to another agent. The specialist takes over and
answers the user directly. Under the hood a handoff is just a tool named
`transfer_to_<agent>` — so the model decides when to hand off exactly the way it decides
to call any tool.

In [ ]:
flights_desk = Agent(
    name="Flights Desk",
    handoff_description="Searches and books flights.",
    instructions=f"{RECOMMENDED_PROMPT_PREFIX}\nYou are the flights desk. Search before quoting. Be concise.",
    model=MODEL,
    tools=[search_flights],
)
hotels_desk = Agent(
    name="Hotels Desk",
    handoff_description="Searches and books hotels.",
    instructions=f"{RECOMMENDED_PROMPT_PREFIX}\nYou are the hotels desk. Search before quoting. Be concise.",
    model=MODEL,
    tools=[search_hotels],
)
visa_desk = Agent(
    name="Visa Desk",
    handoff_description="Answers visa and entry-requirement questions.",
    instructions=(f"{RECOMMENDED_PROMPT_PREFIX}\nYou are the visa desk. Always use check_visa. "
                  "End every answer with: 'Please confirm with the embassy.'"),
    model=MODEL,
    tools=[check_visa],
)

triage = Agent(
    name="Front Desk",
    instructions=(f"{RECOMMENDED_PROMPT_PREFIX}\nYou greet travellers and route them to the right desk. "
                  "Do not answer travel questions yourself — hand off."),
    model=MODEL,
    handoffs=[flights_desk, hotels_desk, visa_desk],
)

result = await Runner.run(triage, "What's the cheapest flight from Bangalore to Goa on 2026-10-02?")
show_run(result)

Read the trace: the Front Desk made **one decision** — hand off — and the Flights Desk
did the work and answered. `result.last_agent` tells you who ended up owning the
conversation.

`RECOMMENDED_PROMPT_PREFIX` is a short block of text telling each agent that it's part of
a multi-agent system and how handoffs work. Print it once to see what it says.

### Multi-turn: who answers the next message?

After a handoff, the specialist owns the conversation. The next turn should start with
**`result.last_agent`**, not the front desk. Otherwise every message goes through triage
again.

In [ ]:
# Let desks send the traveller back to the front desk for off-topic follow-ups.
for desk in (flights_desk, hotels_desk, visa_desk):
    desk.handoffs = [triage]

session = SQLiteSession("handoff-demo")
agent = triage
for turn in ["Flights from Bangalore to Goa on 2026-10-02 please.",
             "The evening one looks good. What time does it leave?",
             "And a hotel in Goa under 6000 a night?"]:
    result = await Runner.run(agent, turn, session=session)
    print(f"\nYOU  -> {turn}")
    show_run(result)
    agent = result.last_agent          # <- the specialist keeps the conversation

Turn 2 went straight to the Flights Desk: no triage hop, one fewer model call. Turn 3
was a hotel question, so the Flights Desk handed off to the Front Desk, which routed it
to the Hotels Desk.

That's a **decentralised** system: control moves around, and nobody is "in charge".

---
# Part 3 — Customising a handoff

`handoff()` wraps an agent with extra behaviour:

- **`on_handoff`** — run your code at the moment of transfer (log it, prefetch data, open a ticket).
- **`input_type`** — make the model state *why* it's handing off, as structured data.
- **`input_filter`** — control what history the specialist inherits.

In [ ]:
from agents.extensions import handoff_filters


class Escalation(BaseModel):
    reason: str
    urgency: str       # "low" | "normal" | "high"


HANDOFF_LOG = []


async def log_visa_escalation(ctx: RunContextWrapper, data: Escalation):
    HANDOFF_LOG.append({"at": datetime.now().strftime("%H:%M:%S"), **data.model_dump()})
    print(f"  [on_handoff] visa desk called in — urgency={data.urgency}: {data.reason}")


triage_v2 = triage.clone(handoffs=[
    flights_desk,
    hotels_desk,
    handoff(
        visa_desk,
        on_handoff=log_visa_escalation,
        input_type=Escalation,
        input_filter=handoff_filters.remove_all_tools,   # visa desk gets a clean history
    ),
])

result = await Runner.run(triage_v2, "I fly to Dubai in 12 days and just realised I might need a visa!")
show_run(result)
print("\nhandoff log:", HANDOFF_LOG)

`input_type` made the model fill in a structured `Escalation` at handoff time, so your
code now knows *why* the transfer happened and how urgent it is. That's the hook for
dashboards, routing to humans, or SLAs.

`remove_all_tools` strips earlier tool calls from the history the visa desk sees:
Week 10's context engineering, applied at the seam between agents.

---
# Part 4 — Handoff or agent-as-tool? The key design decision

Yesterday you used `agent.as_tool()`. Both are ways for agents to use each other. They
are **not** interchangeable.

| | `handoffs=[...]` | `tools=[agent.as_tool()]` |
|---|---|---|
| Who answers the user | the specialist | the original agent |
| Control | transfers | stays with the manager |
| Can use several specialists in one turn | awkward (a chain of hops) | yes, even in parallel |
| Week 10 pattern | **router** / decentralised | **orchestrator** / manager |
| Good for | "who owns this conversation now?" | "I need answers from several experts" |

Same request, both ways:

In [ ]:
REQUEST = "I'm flying Bangalore to Dubai on 2026-10-02. Cheapest flight, a hotel under 8000, and do I need a visa?"

# --- A: handoffs (desks without the way back, so we see a single hop)
router = Agent(
    name="Router",
    instructions=f"{RECOMMENDED_PROMPT_PREFIX}\nRoute to the single best desk.",
    model=MODEL,
    handoffs=[flights_desk.clone(handoffs=[]), hotels_desk.clone(handoffs=[]), visa_desk.clone(handoffs=[])],
)
from agents import MaxTurnsExceeded

print("=== A: HANDOFFS ===")
try:
    a = await Runner.run(router, REQUEST, max_turns=6)
    show_run(a)
except MaxTurnsExceeded as e:
    calls = [getattr(i.raw_item, "name", "?") for i in e.run_data.new_items if type(i).__name__ == "ToolCallItem"]
    print(f"  LOOPED — hit max_turns. The desk it landed on made {len(calls)} tool calls:")
    print("  ", calls)

# --- B: agents as tools
manager = Agent(
    name="Trip Manager",
    instructions="Answer the whole request. Consult every specialist you need, then write one combined answer.",
    model=MODEL,
    tools=[flights_desk.clone(handoffs=[]).as_tool("ask_flights_desk", "Flight search."),
           hotels_desk.clone(handoffs=[]).as_tool("ask_hotels_desk", "Hotel search."),
           visa_desk.clone(handoffs=[]).as_tool("ask_visa_desk", "Visa questions.")],
)
b = await Runner.run(manager, REQUEST)
print("\n=== B: AGENTS AS TOOLS ===")
show_run(b)

Compare the two:

- **A** handed the request to *one* desk, which can only answer its own third of the
  question. Two things can happen, and you may see either:
  - it answers its third and ignores the rest — a **partial answer**; or
  - it **loops**, calling its one tool again and again, because it can't satisfy the
    request and has nowhere to send it. `max_turns` is what stopped it.

  That loop is Week 10's *"looped on one tool"* failure, caused by **design**: a
  dead-end specialist with no hand-back and no instruction for "not my job". The fixes
  are the ones you saw in Part 2: give desks a way back, and tell them what to do with
  the parts they can't handle.
- **B** called all three specialists, probably in parallel, and wrote one combined answer.
  Look at the model-call count, though — it pays for every specialist's loop.

**Rule of thumb:** use a handoff when the *conversation* should move to a specialist.
Use an agent-as-tool when you need a specialist's *answer* and want to stay in charge.
Real systems mix both.

---
# Part 5 — Guardrails

Instructions are *requests*. Guardrails are *checks that run alongside the agent* and can
**trip** — stopping the run with an exception your code catches.

Three kinds, three places:

| Guardrail | Checks | Example |
|---|---|---|
| **input** | the user's message, before (or while) the agent works | off-topic, PII, jailbreaks |
| **output** | the agent's final answer | promises it can't keep |
| **tool** | one tool call's arguments or result | a date in the past |

## 5a. The cheapest guardrail is not an LLM

Passport numbers have a shape, so a regex catches them in microseconds, for free,
deterministically. Try the boring check first.

In [ ]:
PASSPORT = re.compile(r"\b[A-PR-WY][1-9]\d\s?\d{4}[1-9]\b|\b[A-Z]\d{7}\b")


def latest_user_text(user_input) -> str:
    """With a session, guardrails receive the WHOLE history. Check only the new message --
    otherwise one old bad message trips every turn that follows it."""
    if isinstance(user_input, str):
        return user_input
    for item in reversed(user_input):
        if item.get("role") == "user":
            content = item.get("content")
            if isinstance(content, str):
                return content
            return " ".join(part.get("text", "") for part in content if isinstance(part, dict))
    return ""


def recent_turns(user_input, n=4) -> str:
    """The last few user/assistant messages, for checks that need context to judge."""
    if isinstance(user_input, str):
        return f"user: {user_input}"
    lines = []
    for item in user_input:
        role = item.get("role")
        if role not in ("user", "assistant"):
            continue
        content = item.get("content")
        text = content if isinstance(content, str) else " ".join(
            part.get("text", "") for part in (content or []) if isinstance(part, dict))
        if text.strip():
            lines.append(f"{role}: {text.strip()[:300]}")
    return "\n".join(lines[-n:])


@input_guardrail(run_in_parallel=False)     # block BEFORE the agent spends anything
async def no_passport_numbers(ctx, agent, user_input) -> GuardrailFunctionOutput:
    found = PASSPORT.findall(latest_user_text(user_input))
    return GuardrailFunctionOutput(output_info={"matches": len(found)}, tripwire_triggered=bool(found))

## 5b. When you need judgement: a small LLM checker

"Is this about travel?" has no regex. A cheap, fast agent with **structured output** does
the classification, and the guardrail reads its verdict.

In [ ]:
class TopicCheck(BaseModel):
    is_travel_related: bool
    reasoning: str


topic_checker = Agent(
    name="Topic Checker",
    instructions=("You see the end of a conversation. Decide whether the LATEST user message is "
                  "about travel: trips, flights, hotels, visas, destinations, packing, bookings. "
                  "Use earlier turns to resolve references like 'the evening one'. "
                  "Greetings and thanks count as travel-related."),
    model="gpt-4.1-nano",          # the cheapest model that can do this job
    output_type=TopicCheck,
)


@input_guardrail
async def travel_only(ctx, agent, user_input) -> GuardrailFunctionOutput:
    # Topic needs context ("book the evening one" is only travel if you saw the flights),
    # so this check gets the last few turns. The passport check above needs only the new message.
    verdict = (await Runner.run(topic_checker, recent_turns(user_input), context=ctx.context)).final_output
    return GuardrailFunctionOutput(output_info=verdict, tripwire_triggered=not verdict.is_travel_related)

`run_in_parallel` is the trade-off to understand:

- **`True` (the default)** — the guardrail and the agent start together. Fast, but if the
  guardrail trips, the agent may already have spent tokens (and called tools!).
- **`False`** — the guardrail finishes first. Slower, but nothing runs on bad input.

Use `False` for anything whose tools have side effects, or where cost matters.

In [ ]:
guarded_desk = triage.clone(input_guardrails=[no_passport_numbers, travel_only])

for message in ["Help me find a hotel in Goa under 5000.",
                "My passport is Z1234567, please book me to Dubai.",
                "Write my 500-word college essay on climate change.",
                "hi!"]:
    try:
        r = await Runner.run(guarded_desk, message)
        print(f"PASSED   {message!r}\n         -> {r.final_output[:90]!r}\n")
    except InputGuardrailTripwireTriggered as e:
        g = e.guardrail_result
        print(f"BLOCKED  {message!r}\n         by {g.guardrail.get_name()}: {g.output.output_info}\n")

**Look at `"hi!"`.** The checker's instructions say greetings count as travel-related. If
it was blocked anyway, you've just watched a **false positive**: the cheapest model ignored
one line of its instructions. A guardrail is a *classifier*, so it needs what every
classifier needs: a small labelled eval set (Week 10's `evaluate()`), and a fix when it
fails — a stronger model, few-shot examples, or a rule that lets greetings through first.
A guardrail that blocks "hi" is a broken product, not a safe one.

Your code catches the exception and decides what the user sees: a polite refusal, a
redaction prompt ("please don't share passport numbers here"), or a route to a human.
**The guardrail detects. You decide the response.**

## 5c. Output guardrails — catch promises you can't keep

A travel agent must never guarantee a visa. Here's an agent told to be reassuring, and a
guardrail that catches it when it overpromises:

In [ ]:
PROMISES = re.compile(r"\b(guarantee[ds]?|100%|definitely (get|be approved)|no chance of rejection)\b", re.I)


@output_guardrail
async def no_visa_guarantees(ctx, agent, output) -> GuardrailFunctionOutput:
    text = str(output)
    hits = [m.group(0) for m in PROMISES.finditer(text)]
    return GuardrailFunctionOutput(output_info={"phrases": hits}, tripwire_triggered=bool(hits))


eager_agent = Agent(
    name="Eager Visa Helper",
    instructions="Be extremely reassuring. Tell travellers their visa is guaranteed and 100% certain.",
    model=MODEL,
    output_guardrails=[no_visa_guarantees],
)

try:
    r = await Runner.run(eager_agent, "Will my Dubai visa get approved?")
    print("passed:", r.final_output)
except OutputGuardrailTripwireTriggered as e:
    print("BLOCKED before the traveller saw it:", e.guardrail_result.output.output_info)

The answer was generated, then **stopped before it reached the user**. That's the point
of an output guardrail: the last check between the model and your customer.

## 5d. Tool guardrails — validate one call's arguments

Input and output guardrails see whole messages. A **tool input guardrail** sees one tool
call's arguments, and can reject them *before the tool runs* — sending the model a
message instead.

In [ ]:
@tool_input_guardrail
def no_past_dates(data) -> ToolGuardrailFunctionOutput:
    args = json.loads(data.context.tool_arguments or "{}")
    travel_date = args.get("date", "")
    if travel_date and travel_date < TODAY:
        return ToolGuardrailFunctionOutput.reject_content(
            f"{travel_date} is in the past (today is {TODAY}). Ask the traveller for a future date.")
    return ToolGuardrailFunctionOutput.allow()


@function_tool(tool_input_guardrails=[no_past_dates])
def search_flights_checked(origin: str, destination: str, date: str) -> str:
    """Search flights between two airports on a date (YYYY-MM-DD), cheapest first."""
    hits = [f for f in FLIGHTS if f["from"] == origin.upper() and f["to"] == destination.upper() and f["date"] == date]
    return json.dumps(hits) if hits else "No flights found."


probe = Agent(name="Flights Desk", model=MODEL, tools=[search_flights_checked],
              instructions="Search flights as asked.")
r = await Runner.run(probe, "Flights from Bangalore to Goa on 2025-03-01?")
show_run(r)

The tool never ran. The model got a plain-English reason and asked for a future date.
Compare that with an exception: this is Week 10's *"return an error the agent can act
on"*, enforced before your code even runs.

---
# Part 6 — Human in the loop: approvals

A booking charges money and can't be undone. Week 10's rule: **gate on irreversibility**.
The SDK's version is one argument: `needs_approval=True`.

When the model calls an approval-gated tool, the run **pauses** and hands you the pending
call. Nothing executes until a human approves.

In [ ]:
@function_tool(needs_approval=True)
def book_flight(flight_id: str, passenger_name: str) -> str:
    """Book a flight for a passenger and charge the card on file.

    Args:
        flight_id: e.g. "QP-1375".
        passenger_name: full name as on the ID.
    """
    flight = next((f for f in FLIGHTS if f["id"] == flight_id.upper()), None)
    if not flight:
        return f"Unknown flight {flight_id}."
    pnr = f"PNR{len(BOOKINGS) + 1:03d}{flight_id.replace('-', '')[:4]}"
    BOOKINGS.append({"pnr": pnr, "flight": flight_id, "passenger": passenger_name, "inr": flight["price_inr"]})
    return f"Booked {flight_id} for {passenger_name}. {pnr}. Charged INR {flight['price_inr']}."


booking_desk = Agent(
    name="Booking Desk",
    instructions=("When the traveller asks to book a specific flight, call book_flight straight away. "
                  "A human approves every booking, so do NOT ask the traveller to confirm first."),
    model=MODEL,
    tools=[search_flights, book_flight],
)

result = await Runner.run(booking_desk, "Book QP-1375 for Asha Rao, please.")
show_run(result)
print("\nfinal_output:", result.final_output)
print("interruptions:", len(result.interruptions))
print("bookings so far:", BOOKINGS)

The run stopped. `final_output` is `None`, `BOOKINGS` is empty, and `interruptions` holds
the pending call with its exact arguments. A human looks at *those* arguments, not at the
model's description of them.

> **Why the instructions matter:** with vaguer instructions, the model often asks
> "shall I book it?" in chat instead of calling the tool, and the approval gate never gets
> a chance. Tell it the gate exists.

In [ ]:
def human_review(pending) -> bool:
    """A stand-in for a Slack button, an admin screen, or an email link."""
    for item in pending:
        args = json.loads(item.raw_item.arguments)
        print(f"\n  APPROVAL NEEDED  {item.raw_item.name}")
        for k, v in args.items():
            print(f"    {k:<15} {v}")
    return input("  Approve? [y/N] ").strip().lower() == "y"


state = result.to_state()
if human_review(result.interruptions):
    for item in result.interruptions:
        state.approve(item)
else:
    for item in result.interruptions:
        state.reject(item, rejection_message="The traveller's manager declined this booking.")

resumed = await Runner.run(booking_desk, state)      # <- resume from where it paused
show_run(resumed)
print("\nbookings:", BOOKINGS)

Answer `y` and it books. Answer `n` and the model is told the booking was declined, and
explains that to the traveller instead of retrying.

### Approval can arrive hours later

Real approvals don't happen in the same Python process. The paused run **serialises to a
string**: store it in a database, send the approver a link, and resume whenever they click.

In [ ]:
paused = await Runner.run(booking_desk, "Book 6E-512 for Ravi Kumar.")
Path("pending_booking.json").write_text(paused.to_state().to_string())
print("saved paused run:", Path("pending_booking.json").stat().st_size, "bytes")
print("...the approver clicks a link hours later, in a different process...\n")

restored = await RunState.from_string(booking_desk, Path("pending_booking.json").read_text())
for item in restored.get_interruptions():
    print("  approving:", item.raw_item.name, item.raw_item.arguments)
    restored.approve(item)

done = await Runner.run(booking_desk, restored)
print("\n" + done.final_output)
print("bookings:", [b["pnr"] for b in BOOKINGS])

### Not every call needs a human: conditional approval

`needs_approval` can be a function. Here, hotels under ₹8,000 a night book instantly and
anything pricier waits for a human. That's the approval gate from Week 10's
human-in-the-loop slide: put humans where the risk is, not everywhere.

In [ ]:
HOTEL_APPROVAL_OVER_INR = 8000


async def pricey_hotel(ctx, params, call_id) -> bool:
    return params.get("price_per_night_inr", 0) > HOTEL_APPROVAL_OVER_INR


@function_tool(needs_approval=pricey_hotel)
def book_hotel(hotel_id: str, nights: int, price_per_night_inr: int) -> str:
    """Book a hotel for a number of nights at the quoted nightly price."""
    BOOKINGS.append({"pnr": f"HTL{len(BOOKINGS) + 1:03d}", "hotel": hotel_id, "nights": nights,
                     "inr": nights * price_per_night_inr})
    return f"Booked {hotel_id} for {nights} nights. Total INR {nights * price_per_night_inr}."


hotel_booker = Agent(name="Hotel Booker", model=MODEL, tools=[search_hotels, book_hotel],
                     instructions="Book exactly what the traveller asks, straight away. A human reviews expensive bookings.")

for ask in ["Book Fontainhas Heritage Stay (H-GOA-2) in Goa for 2 nights at 3800.",
            "Book Marina View Hotel (H-DXB-1) in Dubai for 2 nights at 11200."]:
    r = await Runner.run(hotel_booker, ask)
    status = "PAUSED for approval" if r.interruptions else "booked instantly"
    print(f"{status:<22} <- {ask}")

---
# Part 7 — Capstone: the Travel Concierge v2

> ### ⚠️ The trap we fell into while building this
> **Input guardrails only run on the first agent of each run.** Part 2 taught you to
> continue each turn from `result.last_agent`. Do that with guardrails on the Front Desk
> only, and from turn 2 onwards every message starts at a *desk* — the guardrails never
> run. Our first version of this capstone let a passport number and a homework request
> straight through, while the code *looked* protected.
>
> The fix below: **every agent that can start a turn carries the same guardrails.** Put
> the check wherever a turn can begin, not where you'd like conversations to begin.
>
> ### ⚠️ …and two more that sessions cause
> 1. **A guardrail sees the whole session history**, not just the new message. Check the
>    whole thing, and one old passport number blocks every later turn. Check *only* the
>    new message, and "book the evening one" looks off-topic. So each check gets the
>    scope it needs: the passport regex reads `latest_user_text()`, and the topic checker
>    reads `recent_turns()`.
> 2. **A blocked message is still saved to the session.** The guardrail stopped the
>    agent, but the passport number was already in storage. *Blocking is not the same as
>    not storing.* The handler below removes it with `session.pop_item()`.

Everything from both days in one system:

- a **Front Desk** with input guardrails (no passport numbers, travel only),
- **Flights, Hotels and Visa desks**, reached by handoff and able to hand back,
- **bookings behind human approval**,
- a **visa output guardrail**,
- **memory** across turns, and one **trace** for the whole conversation.

In [ ]:
GUARDS = [no_passport_numbers, travel_only]
BOOKINGS.clear()                         # start the capstone with an empty booking system

flights_v2 = Agent(
    name="Flights Desk", handoff_description="Searches and books flights.", model=MODEL,
    instructions=(f"{RECOMMENDED_PROMPT_PREFIX}\nFlights desk. Search before quoting. When asked to book, "
                  "call book_flight straight away; a human approves it. Hand back to the Front Desk for "
                  "anything that isn't about flights."),
    tools=[search_flights_checked, book_flight],
    input_guardrails=GUARDS,
)
hotels_v2 = Agent(
    name="Hotels Desk", handoff_description="Searches and books hotels.", model=MODEL,
    instructions=(f"{RECOMMENDED_PROMPT_PREFIX}\nHotels desk. Search before quoting. When asked to book, "
                  "call book_hotel straight away. Hand back to the Front Desk for anything else."),
    tools=[search_hotels, book_hotel],
    input_guardrails=GUARDS,
)
visa_v2 = Agent(
    name="Visa Desk", handoff_description="Visa and entry requirements.", model=MODEL,
    instructions=(f"{RECOMMENDED_PROMPT_PREFIX}\nVisa desk. Always use check_visa. Never promise approval. "
                  "End with: 'Please confirm with the embassy.' Hand back for anything else."),
    tools=[check_visa],
    input_guardrails=GUARDS,
    output_guardrails=[no_visa_guarantees],
)
front_desk = Agent(
    name="Front Desk", model=MODEL,
    instructions=(f"{RECOMMENDED_PROMPT_PREFIX}\nYou are the front desk of a travel concierge. Today is {TODAY}. "
                  "Route every travel request to the right desk. Only answer greetings yourself."),
    handoffs=[flights_v2, hotels_v2, visa_v2],
    input_guardrails=GUARDS,
)
for desk in (flights_v2, hotels_v2, visa_v2):
    desk.handoffs = [front_desk]
print("concierge v2 assembled")

In [ ]:
async def concierge(message, state):
    """One turn: guardrails, handoffs, approvals and memory. Returns the updated state."""
    print(f"\nYOU   {message}")
    try:
        result = await Runner.run(state["agent"], message, session=state["session"], max_turns=12)
        while result.interruptions:                              # approvals, possibly several
            pending = result.to_state()
            ok = human_review(result.interruptions)
            for item in result.interruptions:
                pending.approve(item) if ok else pending.reject(item, rejection_message="Declined by the traveller.")
            result = await Runner.run(result.last_agent, pending, session=state["session"], max_turns=12)
        print(f"DESK  [{result.last_agent.name}] {result.final_output}")
        state["agent"] = result.last_agent
    except InputGuardrailTripwireTriggered as e:
        await state["session"].pop_item()     # the blocked message was already stored -- remove it
        name = e.guardrail_result.guardrail.get_name()
        msg = ("Please don't share passport numbers in chat — our booking form collects them securely."
               if name == "no_passport_numbers" else "I can only help with travel.")
        print(f"DESK  [guardrail: {name}] {msg}")
    except OutputGuardrailTripwireTriggered:
        print("DESK  [guardrail] Sorry, I can't make promises about visa outcomes. Please check with the embassy.")
    return state


chat = {"agent": front_desk, "session": SQLiteSession("asha-v2")}
with trace("Concierge v2 — Asha's Goa trip"):
    for message in ["Hi! I'm Asha, planning a Goa trip from Bangalore on 2026-10-02.",
                    "What flights are there? I prefer evenings.",
                    "Book the evening one for Asha Rao.",
                    "My passport number is Z1234567 if you need it.",
                    "Now a hotel in Goa under 6000 a night for 3 nights — book the cheapest.",
                    "Can you also help me with my maths homework?"]:
        chat = await concierge(message, chat)

print("\nBOOKINGS:")
for b in BOOKINGS:
    print("  ", b)

Read that transcript for four things:

1. **Handoffs** — which desk answered each turn, and the hand-back when the topic changed.
2. **Approval** — the flight booking paused for you. The cheap hotel didn't.
3. **Input guardrails** — the passport number and the homework never reached a desk.
4. **Memory** — "the evening one" and "the cheapest" only work because of the session.

Open the trace at **https://platform.openai.com/traces**. The whole conversation is one
workflow, with every handoff, guardrail and tool call visible.

---
## Where this leaves us

You now have every primitive in the SDK: **Agent, Runner, tools, handoffs, guardrails,
sessions, tracing**, plus approvals. Next: the same concierge, with a voice
(`03_voice_concierge.ipynb`).

## Exercises

**1. A refunds desk.** Add a desk whose `cancel_booking` tool needs approval only when the
cancellation fee is above zero.

**2. Make triage cheaper.** Use `gpt-4.1-nano` for the Front Desk, since all it does is
route. Does routing quality hold? Measure it with the `evaluate()` harness from Week 10.

**3. Guardrail in parallel.** Flip `no_passport_numbers` to `run_in_parallel=True` and
send the passport message again. Did any desk start working before it tripped? Check the
trace.

**4. Redact instead of block.** Rewrite the passport guardrail to *mask* the number
(`Z1******`) and let the request through. Which is better for the traveller?

**5. Manager version.** Rebuild the capstone with agents-as-tools instead of handoffs.
Which one handles "flight, hotel and visa in one message" better? Which is cheaper?